# ML-04 — Search Intelligence Data Contract

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*
### Data contract

1. **What one row means:** One row in my final feature frame represents one pseudonymized content item belonging to one pseudonymized client. The raw warehouse table has a daily grain: one row per report date × client × content item.

2. **Tables used:** I use `fact_content_daily_performance`, restricted to the March 2026 partition. I do not join the query table because its fixed 90-day window could overlap an outcome period.

3. **Time window:** Features use observations from 2026-03-01 through 2026-03-15. The decision moment is 2026-03-16. The outcome window is 2026-03-16 through 2026-03-31.

4. **What I predict/rank:** I predict whether visibility declines during the outcome period. The proxy label is 1 when impressions during 2026-03-24–31 are more than 20% lower than impressions during 2026-03-16–23. Predicted probability becomes the score used to rank content items for review.

5. **Deliberate exclusion:** I exclude all outcome-window measurements from the feature set. I also exclude the June `_sample` table because June 2026 is the sealed final test month, not a development month.

In [8]:
%pip -q install -U duckdb huggingface_hub scikit-learn

import os
import getpass
import duckdb
import pandas as pd
import numpy as np
from IPython.display import display

# Token order: environment variable -> Colab Secret -> hidden prompt.
# In Colab, create a Secret named HF_TOKEN and enable notebook access.
HF_TOKEN = os.environ.get("HF_TOKEN")

if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception:
        HF_TOKEN = None

if not HF_TOKEN:
    HF_TOKEN = getpass.getpass("Enter your Hugging Face READ token: ")

if not HF_TOKEN:
    raise ValueError("HF_TOKEN was not provided.")

con = duckdb.connect()

# The token is used in memory only and is never printed.
safe_token = HF_TOKEN.replace("'", "''")
con.execute(
    f"CREATE OR REPLACE SECRET hf "
    f"(TYPE huggingface, TOKEN '{safe_token}')"
)

REL = "hf://datasets/FlyRank/internship-warehouse"

# Use the March partition directly instead of scanning all 79M rows.
MARCH_FACT = (
    f"read_parquet("
    f"'{REL}/fact_content_daily_performance/month=2026-03/*.parquet'"
    f")"
)

print("DuckDB connected to the March 2026 warehouse partition.")

DuckDB connected to the March 2026 warehouse partition.


## 2. Fields: feature / label / context / excluded
### Field classification

**Features — known before the 2026-03-16 decision moment**

- `log_impressions_feature`: log-transformed impressions from March 1–15.
- `ctr_feature`: clicks divided by impressions during March 1–15.
- `avg_position_feature`: average observed search position during March 1–15.
- `active_days_feature`: number of March 1–15 days with at least one impression.
- `position_std_feature`: variation in search position during March 1–15.

**Label/proxy**

- `is_future_decline`: 1 when impressions during March 24–31 are less than 80% of impressions during March 16–23; otherwise 0.
- `outcome_imp_first8` and `outcome_imp_last8` are label-construction fields and must never enter the honest feature set.

**Context**

- `client_hash_id`: used only for grouping, joining, and later client-level validation.
- `content_hash_id`: identifies the content item and verifies the final grain.

**Excluded**

- Outcome-window measurements: future information used to construct the label.
- `client_hash_id` and `content_hash_id` as model features: hashes have no predictive meaning.
- GA4 engagement fields: availability is uneven, so this first feature frame uses consistently available GSC signals.
- June `_sample` data: it is the sealed test month.

In [9]:
field_contract = pd.DataFrame([
    {
        "field": "log_impressions_feature",
        "bucket": "feature",
        "reason": "Computed only from March 1-15 impressions."
    },
    {
        "field": "ctr_feature",
        "bucket": "feature",
        "reason": "Computed only from March 1-15 clicks and impressions."
    },
    {
        "field": "avg_position_feature",
        "bucket": "feature",
        "reason": "Observed before the March 16 decision moment."
    },
    {
        "field": "active_days_feature",
        "bucket": "feature",
        "reason": "Counts active search days before the decision."
    },
    {
        "field": "position_std_feature",
        "bucket": "feature",
        "reason": "Measures pre-decision position volatility."
    },
    {
        "field": "is_future_decline",
        "bucket": "label/proxy",
        "reason": "Constructed from March 16-31 future impressions."
    },
    {
        "field": "client_hash_id",
        "bucket": "context",
        "reason": "Used for grouping and validation, never as a model feature."
    },
    {
        "field": "content_hash_id",
        "bucket": "context",
        "reason": "Identifies a content item, never used as a model feature."
    },
    {
        "field": "outcome_imp_first8 / outcome_imp_last8",
        "bucket": "excluded",
        "reason": "Future, label-derived information."
    },
    {
        "field": "June sample table",
        "bucket": "excluded",
        "reason": "June 2026 is the sealed final test month."
    }
])

display(field_contract)

,field,bucket,reason
0,log_impressions_feature,feature,Computed only from March 1-15 impressions.
1,ctr_feature,feature,Computed only from March 1-15 clicks and impre...
2,avg_position_feature,feature,Observed before the March 16 decision moment.
3,active_days_feature,feature,Counts active search days before the decision.
4,position_std_feature,feature,Measures pre-decision position volatility.
5,is_future_decline,label/proxy,Constructed from March 16-31 future impressions.
6,client_hash_id,context,"Used for grouping and validation, never as a m..."
7,content_hash_id,context,"Identifies a content item, never used as a mod..."
8,outcome_imp_first8 / outcome_imp_last8,excluded,"Future, label-derived information."
9,June sample table,excluded,June 2026 is the sealed final test month.


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [10]:
# ============================================================
# VERIFICATION QUERY 1 — raw-table grain
# Expected result: an empty dataframe (zero duplicate grains).
# ============================================================

grain_check = con.sql(f"""
    SELECT
        report_date,
        client_hash_id,
        content_hash_id,
        COUNT(*) AS row_count
    FROM {MARCH_FACT}
    GROUP BY
        report_date,
        client_hash_id,
        content_hash_id
    HAVING COUNT(*) > 1
    LIMIT 10
""").df()

print("Query 1 — duplicate raw-grain groups:", len(grain_check))
display(grain_check)


# ============================================================
# VERIFICATION QUERY 2 — slice row count and date span
# ============================================================

slice_summary = con.sql(f"""
    SELECT
        COUNT(*) AS raw_rows,
        COUNT(
            DISTINCT client_hash_id || '|' || content_hash_id
        ) AS content_units,
        MIN(report_date) AS minimum_date,
        MAX(report_date) AS maximum_date
    FROM {MARCH_FACT}
""").df()

print("Query 2 — March slice count and date span")
display(slice_summary)


# ============================================================
# VERIFICATION QUERY 3 — GA4 availability
# The available subquery explicitly filters with IS TRUE.
# ============================================================

availability_summary = con.sql(f"""
    WITH all_march_rows AS (
        SELECT COUNT(*) AS total_rows
        FROM {MARCH_FACT}
    ),
    available_march_rows AS (
        SELECT COUNT(*) AS surviving_rows
        FROM {MARCH_FACT}
        WHERE ga4_data_available IS TRUE
    )
    SELECT
        total_rows,
        surviving_rows,
        ROUND(
            100.0 * surviving_rows / NULLIF(total_rows, 0),
            2
        ) AS percent_surviving
    FROM all_march_rows
    CROSS JOIN available_march_rows
""").df()

print("Query 3 — rows surviving ga4_data_available IS TRUE")
display(availability_summary)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Query 1 — duplicate raw-grain groups: 0


,report_date,client_hash_id,content_hash_id,row_count


Query 2 — March slice count and date span


,raw_rows,content_units,minimum_date,maximum_date
0,9841378,331437,2026-03-01,2026-03-31


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Query 3 — rows surviving ga4_data_available IS TRUE


,total_rows,surviving_rows,percent_surviving
0,9841378,413966,4.21


### Five-feature frame and availability at the decision moment

1. **`log_impressions_feature`** — knowable at the decision moment because it uses only impressions observed from March 1–15.
2. **`ctr_feature`** — knowable at the decision moment because both clicks and impressions were observed by March 15.
3. **`avg_position_feature`** — knowable at the decision moment because it uses only pre-decision search positions.
4. **`active_days_feature`** — knowable at the decision moment because it counts only pre-decision days with impressions.
5. **`position_std_feature`** — knowable at the decision moment because it measures only pre-decision position variation.

Pages must have at least 100 feature-window impressions and at least 100 impressions in the first outcome sub-window. These thresholds reduce highly unstable labels created from very small counts.

**construct the five features**

In [11]:
frame = con.sql(f"""
    WITH page_month AS (
        SELECT
            client_hash_id,
            content_hash_id,

            -- Feature window: March 1-15
            SUM(
                CASE
                    WHEN report_date BETWEEN DATE '2026-03-01'
                                         AND DATE '2026-03-15'
                    THEN gsc_impressions
                    ELSE 0
                END
            ) AS feature_impressions,

            SUM(
                CASE
                    WHEN report_date BETWEEN DATE '2026-03-01'
                                         AND DATE '2026-03-15'
                    THEN gsc_clicks
                    ELSE 0
                END
            ) AS feature_clicks,

            AVG(
                CASE
                    WHEN report_date BETWEEN DATE '2026-03-01'
                                         AND DATE '2026-03-15'
                         AND gsc_impressions > 0
                    THEN gsc_avg_position
                END
            ) AS avg_position_feature,

            COUNT(
                DISTINCT CASE
                    WHEN report_date BETWEEN DATE '2026-03-01'
                                         AND DATE '2026-03-15'
                         AND gsc_impressions > 0
                    THEN report_date
                END
            ) AS active_days_feature,

            STDDEV_POP(
                CASE
                    WHEN report_date BETWEEN DATE '2026-03-01'
                                         AND DATE '2026-03-15'
                         AND gsc_impressions > 0
                    THEN gsc_avg_position
                END
            ) AS position_std_feature,

            -- Future outcome: first eight days
            SUM(
                CASE
                    WHEN report_date BETWEEN DATE '2026-03-16'
                                         AND DATE '2026-03-23'
                    THEN gsc_impressions
                    ELSE 0
                END
            ) AS outcome_imp_first8,

            -- Future outcome: final eight days
            SUM(
                CASE
                    WHEN report_date BETWEEN DATE '2026-03-24'
                                         AND DATE '2026-03-31'
                    THEN gsc_impressions
                    ELSE 0
                END
            ) AS outcome_imp_last8

        FROM {MARCH_FACT}
        GROUP BY
            client_hash_id,
            content_hash_id
    )

    SELECT
        client_hash_id,
        content_hash_id,

        LN(1 + feature_impressions) AS log_impressions_feature,

        feature_clicks::DOUBLE
            / NULLIF(feature_impressions, 0) AS ctr_feature,

        avg_position_feature,
        active_days_feature,
        position_std_feature,

        -- Retained only to construct and audit the label.
        outcome_imp_first8,
        outcome_imp_last8,

        CAST(
            outcome_imp_last8 < 0.80 * outcome_imp_first8
            AS INTEGER
        ) AS is_future_decline

    FROM page_month
    WHERE feature_impressions >= 100
      AND outcome_imp_first8 >= 100
""").df()

FEATURE_COLS = [
    "log_impressions_feature",
    "ctr_feature",
    "avg_position_feature",
    "active_days_feature",
    "position_std_feature",
]

print("Final content-level rows:", len(frame))
print("Number of model features:", len(FEATURE_COLS))
print("Future-decline base rate:", round(frame["is_future_decline"].mean(), 3))

display(frame[FEATURE_COLS + ["is_future_decline"]].head())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Final content-level rows: 62535
Number of model features: 5
Future-decline base rate: 0.231


,log_impressions_feature,ctr_feature,avg_position_feature,active_days_feature,position_std_feature,is_future_decline
0,5.393628,0.004566,3.737399,15,2.467964,0
1,7.309881,0.000000,6.156643,14,1.103299,0
2,6.025866,0.002421,4.390322,15,1.316455,0
3,5.634790,0.003584,9.961993,15,4.904767,0
4,6.701960,0.000000,8.448104,14,2.635988,0


**deliberate leakage experiment**

In [12]:
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

model_data = frame.copy()

train_index, test_index = train_test_split(
    np.arange(len(model_data)),
    test_size=0.25,
    random_state=42,
    stratify=model_data["is_future_decline"]
)

def calculate_auc(columns):
    model = make_pipeline(
        SimpleImputer(strategy="median"),
        RandomForestClassifier(
            n_estimators=150,
            min_samples_leaf=5,
            random_state=42,
            n_jobs=-1
        )
    )

    model.fit(
        model_data.loc[train_index, columns],
        model_data.loc[train_index, "is_future_decline"]
    )

    probabilities = model.predict_proba(
        model_data.loc[test_index, columns]
    )[:, 1]

    return roc_auc_score(
        model_data.loc[test_index, "is_future_decline"],
        probabilities
    )


# Honest score: five pre-decision features only.
honest_auc = calculate_auc(FEATURE_COLS)

# ------------------------------------------------------------
# DELIBERATE TRAP
# This ratio uses the future outcome window that defines the label.
# It is unavailable at the decision moment and therefore leakage.
# ------------------------------------------------------------

model_data["leaky_future_ratio"] = (
    model_data["outcome_imp_last8"]
    / model_data["outcome_imp_first8"]
)

leaky_auc = calculate_auc(
    FEATURE_COLS + ["leaky_future_ratio"]
)

print(f"Honest ROC AUC: {honest_auc:.3f}")
print(f"Leaky ROC AUC:  {leaky_auc:.3f}")
print("The leaky score is invalid because it uses future label information.")

# Delete the leakage and retain only the honest result.
model_data.drop(columns=["leaky_future_ratio"], inplace=True)
FINAL_FEATURES = FEATURE_COLS.copy()
final_retained_auc = honest_auc

assert "leaky_future_ratio" not in model_data.columns
assert len(FINAL_FEATURES) == 5

print(f"Final retained honest ROC AUC: {final_retained_auc:.3f}")
print("Final retained features:", FINAL_FEATURES)

Honest ROC AUC: 0.665
Leaky ROC AUC:  1.000
The leaky score is invalid because it uses future label information.
Final retained honest ROC AUC: 0.665
Final retained features: ['log_impressions_feature', 'ctr_feature', 'avg_position_feature', 'active_days_feature', 'position_std_feature']


## 4. Data limits
The data has uneven history across clients, so some content items have shorter observation periods than others. Early rows may contain only GSC data because GA4 tracking had not started, and overlapping feature and outcome windows could cause leakage. Therefore, the results support ranking pages for review but cannot prove that refreshing a page will improve its performance.


In [13]:
ga4_survival_pct = availability_summary.loc[0, "percent_surviving"]

print(f"Rows with GA4 data available: {ga4_survival_pct}%")
print(
    "Limitation confirmed: GA4 coverage is incomplete, "
    "so this feature frame uses GSC signals only."
)

Rows with GA4 data available: 4.21%
Limitation confirmed: GA4 coverage is incomplete, so this feature frame uses GSC signals only.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.